In [ ]:
!pip install -q executorch huggingface_hub

In [ ]:
!pip list 2>/dev/null | grep -i -E "^(torch|torchao|torchvision|executorch|pytorch-tokenizers) "

In [ ]:
from google.colab import userdata
from huggingface_hub import snapshot_download

path = snapshot_download(
    "meta-llama/Llama-3.2-1B-Instruct",
    allow_patterns=["original/*"],
    token=userdata.get("HF_TOKEN"),
    local_dir="llama",
)
print("done", path)

In [ ]:
cfg = """
base:
  model_class: llama3_2
  checkpoint: llama/original/consolidated.00.pth
  params: llama/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: False
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
backend:
  xnnpack:
    enabled: True
    extended_ops: True
"""
open("export.yaml", "w").write(cfg)

In [ ]:
!python -m executorch.extension.llm.export.export_llm --config export.yaml
!ls -la *.pte

In [ ]:
!find / -name "*.pte" -size +1M 2>/dev/null
from google.colab import files

In [ ]:
files.download("llama3_2.pte")

In [ ]:
!find / -name "tokenizer.model" 2>/dev/null

In [ ]:
from google.colab import files

files.download("tokenizer.model")